In [119]:
import requests

import pandas as pd

from datetime import datetime

API_KEY = "TgIjc2C35ULxg5jDTWCYCIuRXGGBe09YMC6fRz1W"

url = F"https://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-08&api_key={API_KEY}"

all_asteroids = []

target = 10000

while url and len(all_asteroids) < target:
    response = requests.get(url)

    data = response.json()
    neo_data = data["near_earth_objects"]

    for date, asteroids in neo_data.items():
        for asteroid in asteroids:
            asteroid_data = {
                'id':asteroid['id'],
                'neo_ref_id':asteroid['neo_reference_id'],
                'name':asteroid['name'],
                'absolute_magnitude_h':asteroid['absolute_magnitude_h'],
                'estimated_diameter_min_km':asteroid['estimated_diameter']['kilometers']['estimated_diameter_min'],
                'estimated_diameter_max_km':asteroid['estimated_diameter']['kilometers']['estimated_diameter_max'],
                'is_potentially_hazardous':asteroid['is_potentially_hazardous_asteroid'],
                'close_approach_date':(datetime.strptime(asteroid['close_approach_data'][0]['close_approach_date'],'%Y-%m-%d').strftime("%Y-%m-%d")),
                'relative_velocity_(km/h)':float(asteroid['close_approach_data'][0]['relative_velocity']['kilometers_per_hour']),
                'astronomical_unit':float(asteroid['close_approach_data'][0]['miss_distance']['astronomical']),
                'lunar_distance':float(asteroid['close_approach_data'][0]['miss_distance']['lunar']),
                'miss_distance_km':float(asteroid['close_approach_data'][0]['miss_distance']['kilometers']),
                'orbiting_body':asteroid['close_approach_data'][0]['orbiting_body']
            }
            all_asteroids.append(asteroid_data)

            
            
            #stops if we reach the target
            if len(all_asteroids)>=10000:
                break
        if len(all_asteroids)>=10000:
                break

url = data['links']['next']
    



In [65]:
all_asteroids

[{'id': '2415949',
  'neo_ref_id': '2415949',
  'name': '415949 (2001 XY10)',
  'absolute_magnitude_h': 19.37,
  'estimated_diameter_min_km': 0.3552670883,
  'estimated_diameter_max_km': 0.7944013596,
  'is_potentially_hazardous': False,
  'close_approach_date': '2024-01-02',
  'relative_velocity_(km/h)': 57205.8951204341,
  'astronomical_unit': 0.3372535274,
  'lunar_distance': 131.1916221586,
  'miss_distance_km': 50452409.349026635,
  'orbiting_body': 'Earth'},
 {'id': '3160747',
  'neo_ref_id': '3160747',
  'name': '(2003 SR84)',
  'absolute_magnitude_h': 26.0,
  'estimated_diameter_min_km': 0.0167708462,
  'estimated_diameter_max_km': 0.0375007522,
  'is_potentially_hazardous': False,
  'close_approach_date': '2024-01-02',
  'relative_velocity_(km/h)': 38589.054833182,
  'astronomical_unit': 0.1323425924,
  'lunar_distance': 51.4812684436,
  'miss_distance_km': 19798169.933318187,
  'orbiting_body': 'Earth'},
 {'id': '3309828',
  'neo_ref_id': '3309828',
  'name': '(2005 YQ96)',
 

In [67]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.4 MB 2.8 MB/s eta 0:00:06
   -- ------------------------------------- 1.0/16.4 MB 3.6 MB/s eta 0:00:05
   --- ------------------------------------ 1.6/16.4 MB 2.9 MB/s eta 0:00:06
   ----- ---------------------------------- 2.4/16.4 MB 2.7 MB/s eta 0:00:06
   ------- -------------------------------- 2.9/16.4 MB 2.8 MB/s eta 0:00:05
   -------- ------------------------------- 3.4/16.4 MB 2.7 MB/s eta 0:00:05
   -------- ------------------------------- 3.7/16.4 MB 2.6 MB/s eta 0:00:05
   -------- ------------------------------- 3.7/16.4 MB 2.6 MB/s eta 0:00:05
   ---------- ----------------------------- 4.2/16.4 MB 2.2 MB/s eta 0:00:06
   ---------- ----------------------------- 4.5/16.4 MB 2.1 MB/s eta 0:00:06
   ----------- ---------------------------- 4.7/16.4 MB 2.0 MB/s eta 0:00:06
   ------------ --------------------------- 5.0/16.4 MB 1.9 MB/s eta 0:00:06
   ---

In [7]:
import mysql.connector as db

In [15]:
import pandas as pd

In [9]:
connection = db.connect(
    host = 'localhost',
    user = 'root',
    password ='Shaffie0000' ,
    database = 'astro'
)

In [11]:
curr = connection.cursor()

In [321]:
curr.execute(
    """
    CREATE TABLE asteroids(
    id INT,
    name VARCHAR(100),
    absolute_magnitude_h FLOAT,
    estimated_diameter_min_km FLOAT,
    estimated_diameter_max_km FLOAT,
    is_potentially_hazardous BOOL
    );
    
    """
)


ProgrammingError: 1050 (42S01): Table 'asteroids' already exists

In [453]:
insert_query = "insert into asteroids values(%s,%s,%s,%s,%s,%s)"

for i in all_asteroids:
    id = int(i['id'])
    name = i['name']
    magnitude = float(i['absolute_magnitude_h'])
    dia_min = float(i['estimated_diameter_min_km'])
    dia_max = float(i['estimated_diameter_max_km'])
    hazard = i['is_potentially_hazardous'] 
    values = [(id,name,magnitude,dia_min,dia_max,hazard)]

    curr.executemany(insert_query,values)
    connection.commit()
    
    

In [1]:
query = "select * from asteroids"
df = pd.read_sql(query, connection)
df


NameError: name 'pd' is not defined

In [445]:
curr.execute(
    '''
    create table CLOSE_APPROACH (
    neo_reference_id INT,
    close_approach_date DATE,
    relative_velocity_km_per_hr FLOAT,
    astronomical_unit FLOAT,
    lunar_distance FLOAT,
    miss_distance_km FLOAT,
    orbiting_body VARCHAR(100)
    );
    
    '''
)

ProgrammingError: 1050 (42S01): Table 'close_approach' already exists

In [452]:
insert_query = "insert into CLOSE_APPROACH values(%s,%s,%s,%s,%s,%s,%s)"

for i in all_asteroids:
    neo_id = int(i['neo_ref_id'])
    close_app_date =  i['close_approach_date']
    velocity = float(i['relative_velocity_(km/h)'])
    atronomical_unit = float(i['astronomical_unit'])
    lunar_distance = float(i['lunar_distance'])
    miss_distance = float( i['miss_distance_km']) 
    orbiting_body = i['orbiting_body']
    values = [(neo_id,close_app_date,velocity,atronomical_unit,lunar_distance,miss_distance,orbiting_body)]

    curr.executemany(insert_query,values)
    connection.commit()
    

In [331]:
query = "select * from CLOSE_APPROACH"
df = pd.read_sql(query, connection)
df

C:\Users\Appu\AppData\Local\Temp\ipykernel_25468\3414721648.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,neo_reference_id,close_approach_date,relative_velocity_km_per_hr,astronomical_unit,lunar_distance,miss_distance_km,orbiting_body
0,2415949,2024-01-02,57205.9,0.337254,131.1920,50452400.0,Earth
1,3160747,2024-01-02,38589.1,0.132343,51.4813,19798200.0,Earth
2,3309828,2024-01-02,56413.0,0.167013,64.9679,24984700.0,Earth
3,3457842,2024-01-02,21891.1,0.492051,191.4080,73609800.0,Earth
4,3553062,2024-01-02,31469.0,0.235802,91.7271,35275500.0,Earth
...,...,...,...,...,...,...,...
9995,54448604,2024-01-08,27845.1,0.198545,77.2339,29701900.0,Earth
9996,2199003,2024-01-07,55047.0,0.199535,77.6189,29849900.0,Earth
9997,2434188,2024-01-07,121814.0,0.204314,79.4781,30564900.0,Earth
9998,3398095,2024-01-07,63326.1,0.191767,74.5972,28687900.0,Earth


#############QUERIES

In [512]:
#Count how many times each asteroid has approached Earth

query2 = """
select neo_reference_id, count(*) as approach_count
from CLOSE_APPROACH
group by neo_reference_id;
"""
curr.execute(query2)
results = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results, columns = columns)
print(df)


     neo_reference_id  approach_count
0             2415949             154
1             3160747             154
2             3309828             154
3             3457842             154
4             3553062             154
..                ...             ...
125          54418496             152
126          54418797             152
127          54419021             152
128          54419611             152
129          54514905             152

[130 rows x 2 columns]


In [514]:
#List top 10 fastest asteroids

query1 = """
select DISTINCT neo_reference_id,relative_velocity_km_per_hr
from CLOSE_APPROACH
order by relative_velocity_km_per_hr desc
LIMIT 10
"""

curr.execute(query1)
results1 = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results1, columns = columns)
print(df)

   neo_reference_id  relative_velocity_km_per_hr
0           3645001                     136268.0
1           2434188                     121814.0
2           3655366                      95388.9
3           3724393                      91415.4
4          54151575                      89673.0
5           2450293                      85384.4
6           2304640                      82917.4
7           3789479                      77947.8
8          54419807                      77405.1
9          54162245                      77270.4


In [516]:
#Average velocity of each asteroid over multiple approaches
query3 = """
select neo_reference_id,relative_velocity_km_per_hr
from CLOSE_APPROACH
"""

curr.execute(query3)
results1 = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results1, columns = columns)

average_velocities = df.groupby("neo_reference_id")["relative_velocity_km_per_hr"].mean().reset_index()
average_velocities = average_velocities.rename(columns={"relative_velocity_km_per_hr": "average_velocity"})
print(average_velocities)

     neo_reference_id  average_velocity
0             2168318           31986.5
1             2199003           55047.0
2             2304640           82917.4
3             2415949           57205.9
4             2434188          121814.0
..                ...               ...
125          54445206           17946.4
126          54448604           27845.1
127          54467128           21162.8
128          54513047           27926.0
129          54514905           65934.8

[130 rows x 2 columns]


In [518]:
#Find potentially hazardous asteroids that have approached Earth more than 3 times
query4 = """
select id, is_potentially_hazardous
from asteroids
"""
curr.execute(query4)
results = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results, columns = columns)

hazardous_df = df[df["is_potentially_hazardous"] == True]
approach_counts = hazardous_df.groupby("id").size().reset_index(name="approach_count")
hazardous_frequent = approach_counts[approach_counts["approach_count"] >3]
print(hazardous_frequent)


          id  approach_count
0    2168318             308
1    2199003             308
2    2434188             308
3    2450293             308
4    2585310             308
5    2613286             308
6    2669051             308
7    3102728             308
8    3309828             308
9    3440771             308
10   3608936             308
11   3794988             308
12  54514905             304


In [522]:
#Find the month with the most asteroid approaches
query2 = """
select close_approach_date
from CLOSE_APPROACH
"""
curr.execute(query2)
results = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results, columns = columns)

df['close_approach_date'] = pd.to_datetime(df['close_approach_date'])
df['month'] = df['close_approach_date'].dt.month

monthly_counts = df['month'].value_counts().sort_values(ascending=False).reset_index()
monthly_counts.columns = ['month', 'approach_count']

print("Month with the most approaches:")
print(monthly_counts.head(1))



Month with the most approaches:
   month  approach_count
0      1           20000


In [554]:
#Get the asteroid with the fastest ever approach speed

query1 = """
select DISTINCT neo_reference_id,relative_velocity_km_per_hr
from CLOSE_APPROACH
order by relative_velocity_km_per_hr desc
LIMIT 1
"""

curr.execute(query1)
results1 = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results1, columns = columns)
print(df)

   neo_reference_id  relative_velocity_km_per_hr
0           3645001                     136268.0


In [526]:
#Sort asteroids by maximum estimated diameter (descending)

query1 = """
select DISTINCT id,estimated_diameter_max_km
from asteroids
order by estimated_diameter_max_km desc
"""

curr.execute(query1)
results1 = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results1, columns = columns)
print(df)

           id  estimated_diameter_max_km
0     2199003                   1.386880
1     2450293                   1.018690
2     2168318                   0.981838
3     2434188                   0.903733
4     2304640                   0.871044
..        ...                        ...
125  54212682                   0.014724
126  54418649                   0.010521
127  54278142                   0.010329
128  54373932                   0.005943
129   3787600                   0.003927

[130 rows x 2 columns]


In [528]:
#Asteroids whose closest approach is getting nearer over time(Hint: Use ORDER BY close_approach_date and look at miss_distance)

query1 = """
select DISTINCT neo_reference_id, close_approach_date, miss_distance_km
from CLOSE_APPROACH
ORDER BY neo_reference_id, close_approach_date
"""

curr.execute(query1)
results1 = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results1, columns = columns)

def is_getting_closer(group):
    return group['miss_distance_km'].is_monotonic_decreasing
    
closer_over_time = df.groupby('neo_reference_id').filter(is_getting_closer)
unique_asteroids = closer_over_time['neo_reference_id'].unique()

print("Asteroids whose closest approach is getting nearer over time:")
print(unique_asteroids)

Asteroids whose closest approach is getting nearer over time:
[ 2168318  2199003  2304640  2415949  2434188  2450293  2585310  2591179
  2613286  2668499  2669051  2676480  3102728  3102756  3154493  3160747
  3309828  3387092  3398095  3440771  3457842  3520662  3541505  3553062
  3591616  3591759  3595775  3596030  3599868  3605578  3608936  3623521
  3630638  3645001  3655366  3724393  3751529  3752775  3759280  3759758
  3780672  3781448  3787600  3789479  3794988  3795150  3795154  3797695
  3798020  3825490  3836410  3837605  3842680  3843476  3873324  3986699
  3986746  3989198 54016989 54048870 54100190 54131589 54151575 54162245
 54191244 54200446 54212682 54212701 54232321 54235663 54239839 54244177
 54278142 54293876 54336963 54338714 54339170 54360724 54373932 54376461
 54377575 54382904 54397463 54399994 54406858 54407114 54407115 54415081
 54415862 54416724 54416853 54417653 54417658 54418438 54418496 54418497
 54418498 54418648 54418649 54418650 54418796 54418797 5441901

In [530]:
#List names of asteroids that approached Earth with velocity > 50,000 km/h

query_asteroids = "SELECT distinct id, name FROM ASTEROIDS"
curr.execute(query_asteroids)
asteroid_data = curr.fetchall()
asteroid_df = pd.DataFrame(asteroid_data, columns=[desc[0] for desc in curr.description])

query_approach = "SELECT distinct neo_reference_id, relative_velocity_km_per_hr FROM CLOSE_APPROACH"
curr.execute(query_approach)
approach_data = curr.fetchall()
approach_df = pd.DataFrame(approach_data, columns=[desc[0] for desc in curr.description])

merged_df = pd.merge(approach_df, asteroid_df, left_on="neo_reference_id", right_on="id")

filtered_df = merged_df[merged_df["relative_velocity_km_per_hr"] > 50000]

result_df = filtered_df[["name", "relative_velocity_km_per_hr"]].sort_values(by="relative_velocity_km_per_hr", ascending=False)
print(result_df)



                   name  relative_velocity_km_per_hr
47          (2013 NT11)                     136268.0
117  434188 (2003 AD23)                     121814.0
86            (2013 YD)                      95388.9
22          (2015 OD22)                      91415.4
123           (2021 LA)                      89673.0
32    450293 (2004 LV3)                      85384.4
66    304640 (2006 WW1)                      82917.4
50           (2017 WQ1)                      77947.8
112          (2024 AC3)                      77405.1
74           (2021 MQ1)                      77270.4
13            (2024 AA)                      77155.6
8            (2019 KK5)                      74999.3
88           (2017 YA8)                      74295.4
35           (2022 BR6)                      73098.6
99          (2012 DN31)                      72972.4
105           (2022 AB)                      72258.1
26           (2022 AO2)                      70248.9
25           (2021 SG2)                      6

In [532]:
#Display the name of each asteroid along with the date and miss distance of its closest approach to Earth.

query_asteroids = "SELECT distinct id, name FROM ASTEROIDS"
curr.execute(query_asteroids)
asteroid_data = curr.fetchall()
asteroid_df = pd.DataFrame(asteroid_data, columns=[desc[0] for desc in curr.description])

query_approach = "SELECT distinct neo_reference_id, close_approach_date,miss_distance_km FROM CLOSE_APPROACH"
curr.execute(query_approach)
approach_data = curr.fetchall()
approach_df = pd.DataFrame(approach_data, columns=[desc[0] for desc in curr.description])

merged_df = pd.merge(
    approach_df,
    asteroid_df,
    left_on="neo_reference_id",
    right_on="id",
    how="inner" 
)
result_df = merged_df[["name", "close_approach_date", "miss_distance_km"]]

print(result_df)

                   name close_approach_date  miss_distance_km
0    415949 (2001 XY10)          2024-01-02        50452400.0
1           (2003 SR84)          2024-01-02        19798200.0
2           (2005 YQ96)          2024-01-02        24984700.0
3           (2009 HC21)          2024-01-02        73609800.0
4           (2010 XA11)          2024-01-02        35275500.0
..                  ...                 ...               ...
125          (2023 YV1)          2024-01-07         8449920.0
126           (2024 AM)          2024-01-07         1209360.0
127           (2024 AW)          2024-01-07         4280160.0
128          (2024 AL2)          2024-01-07         1496560.0
129          (2025 AK2)          2024-01-07        28265400.0

[130 rows x 3 columns]


In [534]:
#Count how many approaches happened per month


query2 = """
select close_approach_date
from CLOSE_APPROACH
"""
curr.execute(query2)
results = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results, columns = columns)

df['close_approach_date'] = pd.to_datetime(df['close_approach_date'])
df['month'] = df['close_approach_date'].dt.month_name()

monthly_counts = df['month'].value_counts().sort_values(ascending=False).reset_index()
monthly_counts.columns = ['month', 'approach_count']


print(monthly_counts)

     month  approach_count
0  January           20000


In [536]:
#Find asteroid with the highest brightness (lowest magnitude value)

query_asteroids = "SELECT distinct id,name, absolute_magnitude_h FROM ASTEROIDS"
curr.execute(query_asteroids)
asteroid_data = curr.fetchall()
asteroid_df = pd.DataFrame(asteroid_data, columns=[desc[0] for desc in curr.description])

brightest = asteroid_df.loc[asteroid_df["absolute_magnitude_h"].idxmin()]

print(f"The brightest asteroid is {brightest['name']} with a magnitude of {brightest['absolute_magnitude_h']}.")

The brightest asteroid is 199003 (2005 WJ56) with a magnitude of 18.16.


In [538]:
#Get number of hazardous vs non-hazardous asteroids

query_asteroids = "SELECT distinct id,is_potentially_hazardous FROM ASTEROIDS"
curr.execute(query_asteroids)
asteroid_data = curr.fetchall()
asteroid_df = pd.DataFrame(asteroid_data, columns=[desc[0] for desc in curr.description])

hazardous_count = asteroid_df[asteroid_df["is_potentially_hazardous"] == True].shape[0]
non_hazardous_count = asteroid_df[asteroid_df["is_potentially_hazardous"] == False].shape[0]

print(f"Number of hazardous asteroids: {hazardous_count}")
print(f"Number of non-hazardous asteroids: {non_hazardous_count}")


Number of hazardous asteroids: 13
Number of non-hazardous asteroids: 117


In [540]:
#asteroids that passed closer than the Moon (lesser than 1 LD), along with their close approach date and distance.


query_asteroids = "SELECT distinct id, name FROM ASTEROIDS"
curr.execute(query_asteroids)
asteroid_data = curr.fetchall()
asteroid_df = pd.DataFrame(asteroid_data, columns=[desc[0] for desc in curr.description])

query_approach = "SELECT distinct neo_reference_id, lunar_distance, miss_distance_km,close_approach_date FROM CLOSE_APPROACH"
curr.execute(query_approach)
approach_data = curr.fetchall()
approach_df = pd.DataFrame(approach_data, columns=[desc[0] for desc in curr.description])

approach_df = approach_df[approach_df["miss_distance_km"] < 384400]

merged_df = pd.merge(
    approach_df,
    asteroid_df,
    left_on="neo_reference_id",
    right_on="id",
    how="inner"
)

result_df = merged_df[["name", "close_approach_date", "miss_distance_km"]]

print(result_df)

        name close_approach_date  miss_distance_km
0  (2024 AD)          2024-01-04          242676.0


In [560]:
#asteroids that came within 0.05 AU(astronomical distance)

query_approach = "SELECT distinct neo_reference_id, astronomical_unit, close_approach_date from CLOSE_APPROACH "
curr.execute(query_approach)
approach_data = curr.fetchall()
approach_df = pd.DataFrame(approach_data, columns=[desc[0] for desc in curr.description])

query_asteroids = "SELECT DISTINCT id, name FROM ASTEROIDS"
curr.execute(query_asteroids)
asteroid_data = curr.fetchall()
asteroid_df = pd.DataFrame(asteroid_data, columns=[desc[0] for desc in curr.description])

merged_df = pd.merge(
    approach_df,
    asteroid_df,
    left_on="neo_reference_id",
    right_on="id",
    how="inner")

filtered_df = merged_df[merged_df["astronomical_unit"] < 0.05]

result_df = merged_df[["name", "close_approach_date", "astronomical_unit"]]

print(result_df)

                   name close_approach_date  astronomical_unit
0    415949 (2001 XY10)          2024-01-02           0.337254
1           (2003 SR84)          2024-01-02           0.132343
2           (2005 YQ96)          2024-01-02           0.167013
3           (2009 HC21)          2024-01-02           0.492051
4           (2010 XA11)          2024-01-02           0.235802
..                  ...                 ...                ...
125          (2023 YV1)          2024-01-07           0.056484
126           (2024 AM)          2024-01-07           0.008084
127           (2024 AW)          2024-01-07           0.028611
128          (2024 AL2)          2024-01-07           0.010004
129          (2025 AK2)          2024-01-07           0.188942

[130 rows x 3 columns]


In [544]:
 #range of diameter of asteroids
query_asteroids = "SELECT distinct id, name,estimated_diameter_min_km,estimated_diameter_max_km FROM ASTEROIDS"
curr.execute(query_asteroids)
asteroid_data = curr.fetchall()
asteroid_df = pd.DataFrame(asteroid_data, columns=[desc[0] for desc in curr.description])

asteroid_df['estimated_diameter_min_km'] = pd.to_numeric(asteroid_df['estimated_diameter_min_km'], errors='coerce')
asteroid_df['estimated_diameter_max_km'] = pd.to_numeric(asteroid_df['estimated_diameter_max_km'], errors='coerce')

asteroid_df['diameter_range_km'] = asteroid_df['estimated_diameter_max_km'] - asteroid_df['estimated_diameter_min_km']

print(asteroid_df[['name', 'estimated_diameter_min_km', 'estimated_diameter_max_km', 'diameter_range_km']])


                   name  estimated_diameter_min_km  estimated_diameter_max_km  \
0    415949 (2001 XY10)                   0.355267                   0.794401   
1           (2003 SR84)                   0.016771                   0.037501   
2           (2005 YQ96)                   0.199781                   0.446725   
3           (2009 HC21)                   0.101054                   0.225964   
4           (2010 XA11)                   0.016016                   0.035813   
..                  ...                        ...                        ...   
125          (2023 YV1)                   0.048815                   0.109154   
126           (2024 AM)                   0.007220                   0.016145   
127           (2024 AW)                   0.020824                   0.046563   
128          (2024 AL2)                   0.008883                   0.019863   
129          (2025 AK2)                   0.119277                   0.266710   

     diameter_range_km  
0 

In [550]:
#biggest diameter and shortest diameter

query_asteroids = "SELECT distinct id, name,estimated_diameter_min_km,estimated_diameter_max_km FROM ASTEROIDS"
curr.execute(query_asteroids)
asteroid_data = curr.fetchall()
asteroid_df = pd.DataFrame(asteroid_data, columns=[desc[0] for desc in curr.description])


biggest_asteroid = asteroid_df.loc[asteroid_df['estimated_diameter_max_km'].idxmax()]

smallest_asteroid = asteroid_df.loc[asteroid_df['estimated_diameter_min_km'].idxmin()]

print("Biggest Asteroid:")
print(biggest_asteroid[['name', 'estimated_diameter_max_km']])

print("\nSmallest Asteroid:")
print(smallest_asteroid[['name', 'estimated_diameter_min_km']])

Biggest Asteroid:
name                         199003 (2005 WJ56)
estimated_diameter_max_km               1.38688
Name: 116, dtype: object

Smallest Asteroid:
name                         (2017 UJ2)
estimated_diameter_min_km      0.001756
Name: 49, dtype: object


In [552]:
#Get the asteroid with the slowest ever approach speed

query1 = """
select DISTINCT neo_reference_id,relative_velocity_km_per_hr
from CLOSE_APPROACH
order by relative_velocity_km_per_hr 
LIMIT 1
"""

curr.execute(query1)
results1 = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results1, columns = columns)
print(df)



   neo_reference_id  relative_velocity_km_per_hr
0           3798020                      1909.58


In [556]:
#date with most asteroids
query2 = """
select close_approach_date
from CLOSE_APPROACH
"""
curr.execute(query2)
results = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results, columns = columns)

df['close_approach_date'] = pd.to_datetime(df['close_approach_date'])

date_counts = df['close_approach_date'].value_counts().sort_values(ascending=False).reset_index()
date_counts.columns = ['close_approach_date', 'approach_count']

print("Date with the most approaches:")
print(date_counts.head(1))

Date with the most approaches:
  close_approach_date  approach_count
0          2024-01-03            3696


In [558]:
##date with least asteroids

query2 = """
select close_approach_date
from CLOSE_APPROACH
"""
curr.execute(query2)
results = curr.fetchall()

columns = [desc[0] for desc in curr.description]
df = pd.DataFrame(results, columns = columns)

df['close_approach_date'] = pd.to_datetime(df['close_approach_date'])

date_counts = df['close_approach_date'].value_counts().sort_values(ascending=False).reset_index()
date_counts.columns = ['close_approach_date', 'approach_count']

print("Date with the most approaches:")
print(date_counts.tail(1))

Date with the most approaches:
  close_approach_date  approach_count
7          2024-01-04            1540
